# rq_stock_all_data 数据质量审计

**审计目的**: 检查 `rq_stock_all_data` 本地 Parquet 数据的质量问题  
**审计区间**: 2018-01-02 ~ 2026-07-10  
**审计日期**: 2026-07-12  
**审计原因**: 情绪择时回测中发现 `benchmark_cum` 从2025年9月起全为 NaN

In [ ]:
import polars as pl
import pandas as pd
import os, sys
from datetime import date

sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))
from my_utils.fun import read_day_data, get_data_trading_days

DS = 'rq_stock_all_data'
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 200)

---
## 结论

**发现一个系统性数据缺陷：2025/09/12 ~ 2025/10/17 共20个交易日，全市场 `total_mv` 全部为0。**

这导致依赖市值加权的基准（`benchmark_value_weight_ret`、`benchmark_cum`）从2025年9月起全部 NaN。

其他字段（volume、amount、pct、adj_factor、close）均无异常。

---
## 问题1（🔴 核心）：total_mv 在 2025年9-10月全部为0

从2025-09-12开始，全市场5000+只股票的 `total_mv` 全部变为0，持续到2025-10-17才恢复正常。

In [ ]:
test = read_day_data(date(2025, 9, 1), date(2025, 10, 31),
                     fields=['trading_date','code','total_mv','close','volume','amount'],
                     file_path=DS)

daily = test.group_by('trading_date').agg([
    pl.col('code').count().alias('n'),
    (pl.col('total_mv') == 0).sum().alias('mv_zero'),
    pl.col('total_mv').sum().alias('mv_sum'),
    (pl.col('volume') == 0).sum().alias('vol_zero'),
]).sort('trading_date')

print(f'{"日期":<14} {"n":>5} {"mv=0":>6} {"mv_sum":>16} {"vol=0":>6}  {"标记":}')
print('-' * 70)
bad_days = 0
for row in daily.iter_rows(named=True):
    is_bad = row['mv_zero'] > 1000
    if is_bad:
        bad_days += 1
    flag = '🔴 total_mv=0' if is_bad else '✓'
    print(f"{row['trading_date']!s:<14} {row['n']:>5d} {row['mv_zero']:>6d} {row['mv_sum']:>16.2e} {row['vol_zero']:>6d}  {flag}")

print(f'\n🔴 共 {bad_days} 个交易日所有股票 total_mv=0，区间：2025-09-12 ~ 2025-10-17')
print('   这段时间 close/volume/amount 数据正常，只有 total_mv 异常。')

### 传导链

```
total_mv=0（全市场）
  → (pct/100 * total_mv).sum() = 0, total_mv.sum() = 0
  → daily_value_weight_ret = 0/0 = NaN
  → 对应的4个周度 benchmark_value_weight_ret = product(1+NaN)-1 = NaN
  → next_week_ret = shift(-1) 把 NaN 前移 → ×仓位 = NaN
  → benchmark_cum = cum_prod(1+NaN) → 从此全 NaN
```

**这是数据源的问题，不是代码逻辑错误。**

---
## 问题2（🟢 正常）：volume=0 / amount=0（停牌股）

每天都有少数股票 volume=0，这些是当天停牌/全天无成交的股票。2020年后稳定在每天 ~12只（0.2~0.3%）。

In [ ]:
df = read_day_data(date(2025,1,1), date(2025,12,31),
                   fields=['trading_date','volume'], file_path=DS)

vol = df.group_by('trading_date').agg([
    (pl.col('volume') == 0).sum().alias('vol_zero'),
    pl.col('volume').count().alias('total'),
]).sort('trading_date')

mn = vol['vol_zero'].min()
mx = vol['vol_zero'].max()
md = vol['vol_zero'].median()
ratio = (vol['vol_zero'].cast(pl.Float64) / vol['total'].cast(pl.Float64)).median()

print(f'2025年 volume=0 最少: {mn} 只/天')
print(f'2025年 volume=0 最多: {mx} 只/天')
print(f'2025年 volume=0 中位数: {md} 只/天')
print(f'占比中位数: {ratio*100:.2f}%')
print('\n结论：停牌股占比 < 0.3%，正常。')

---
## 问题3（🟢 正常）：pct 极端值（>40% / <-40%）

针对2025年 pct>40% 或 <-40% 的极端记录，按原因分类。

In [ ]:
df = read_day_data(date(2025,1,1), date(2025,12,31),
                   fields=['code','trading_date','pct','close','open','pre_close',
                           'limit_up','limit_down','is_st','is_suspended'],
                   file_path=DS)

extreme = df.filter((pl.col('pct') > 40) | (pl.col('pct') < -40))
print(f'2025年 pct 极端值总数: {len(extreme)}')

# 类别①：新股首日（该股票之前无数据）
earliest = df.group_by('code').agg(pl.col('trading_date').min().alias('first_seen'))
extreme = extreme.join(earliest, on='code')
extreme = extreme.with_columns(
    (pl.col('trading_date') == pl.col('first_seen')).alias('is_first_day')
)

# 类别②：无涨跌幅限制日（limit_up=0，含退市整理期+新股前5日+恢复上市）
extreme = extreme.with_columns(
    (pl.col('limit_up') == 0).alias('no_limit_day')
)

n1 = extreme['is_first_day'].sum()
n2 = extreme.filter(~pl.col('is_first_day'))['no_limit_day'].sum()
n3 = len(extreme) - n1 - n2

print(f'\n分类结果：')
print(f'  ① 新股首日（之前无数据）:      {n1} 条')
print(f'  ② 其他无涨跌幅限制日:          {n2} 条')
print(f'  ③ 其他（不可解释）:            {n3} 条')

if n3 > 0:
    other = extreme.filter(~pl.col('is_first_day') & ~pl.col('no_limit_day')) \
        .select(['trading_date','code','pct','close','pre_close','is_st','limit_up'])
    print(other.to_pandas().to_string())

print('\n结论：极端 pct 全部可解释为新股首日或特殊交易安排（无涨跌幅限制），不是数据错误。')

---
## 问题4（🟢 正常）：交易日覆盖

对比 `get_data_trading_days()` 交易日历，全部 2066 个交易日都有数据，无缺失。

In [ ]:
all_days = set(get_data_trading_days(date(2018,1,2), date(2026,7,10)))
data_days = set()
for yr in range(2018, 2027):
    sd = date(yr, 1, 1)
    ed = date(yr+1, 1, 1) if yr < 2026 else date(2026, 7, 11)
    for d in read_day_data(sd, ed, fields=['trading_date'], file_path=DS)['trading_date'].unique():
        data_days.add(d)

missing = sorted(all_days - data_days)
print(f'交易日历: {len(all_days)} 天')
print(f'数据覆盖: {len(data_days)} 天')
print(f'缺失: {len(missing)} 天')
if missing:
    print(f'缺失日期: {missing[:10]}...')
else:
    print('✅ 全部覆盖')

---
## 总结

| 问题 | 严重程度 | 说明 |
|------|---------|------|
| **total_mv=0 (2025/09/12~10/17)** | 🔴 | 20个交易日全市场总市值=0，市值加权基准崩坏 |
| volume=0 个别股票 | 🟢 | 停牌股，占比<0.3% |
| pct 极端值 | 🟢 | 新股首日或特殊交易安排（无涨跌幅限制） |
| 交易日覆盖 | 🟢 | 全部2066天无缺失 |

**唯一需要修复的是 total_mv 数据。**

### 修复建议

1. 从米筐重新拉取 2025-09-12 ~ 2025-10-17 的 `total_mv` 数据
2. 临时规避：在回测脚本中给 `daily_value_weight_ret` 加回退：
   ```python
   pl.when(pl.col('total_mv').sum() > 0)
     .then((pl.col('pct') / 100.0 * pl.col('total_mv')).sum() / pl.col('total_mv').sum())
     .otherwise(pl.col('pct').mean() / 100.0)
     .alias('daily_value_weight_ret')
   ```